## Step 1: Load and clean the data

(Same cleaning function from notebook 02 - reused here.)


In [1]:
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\{\{.*?\}\}", " ", text)
    text = re.sub(r"\{.*?\}", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = pd.read_csv("../data/raw/tickets.csv")
df = df[["instruction", "category"]].dropna()
df = df.drop_duplicates(subset=["instruction"])
df.columns = ["text", "label"]
df["text"] = df["text"].apply(clean_text)

print("Final dataset shape after cleaning + dedup:", df.shape)
df.head()

Final dataset shape after cleaning + dedup: (24635, 2)


,text,label
0,question about cancelling order,ORDER
1,i have a question about cancelling oorder,ORDER
2,i need help cancelling puchase,ORDER
3,i need to cancel purchase,ORDER
4,i cannot afford this order cancel purchase,ORDER


## Step 2: Understand "stratified" splitting

If we split randomly without care, we might accidentally end up with almost
no REFUND examples in the test set just by bad luck - especially for smaller
categories like CANCEL.

**Stratified splitting** makes sure each split (train/val/test) has the SAME
proportion of each category as the original data. `sklearn`'s `train_test_split`
does this automatically when we pass `stratify=df["label"]`.


In [3]:
# First split: 80% train, 20% temporary (which we'll split again below)
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["label"],
    random_state=42   # fixes randomness so results are reproducible every run
)

print("Train size:", len(train_df))
print("Temp size (will become val+test):", len(temp_df))

Train size: 17244
Temp size (will become val+test): 7391


## Step 3: Split the remaining 20% into validation and test (10% each)


In [4]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,   # half of the remaining 20% = 10% of total
    stratify=temp_df["label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

Train size: 17244
Validation size: 3695
Test size: 3696


## Step 4: Verify the split preserved category proportions

Let's confirm each set has a similar percentage breakdown per category -
this proves stratification worked correctly.


In [5]:
print("Train proportions:")
print((train_df["label"].value_counts(normalize=True) * 100).round(1))

print("\nTest proportions:")
print((test_df["label"].value_counts(normalize=True) * 100).round(1))

Train proportions:
label
ACCOUNT         22.1
ORDER           12.9
REFUND          10.6
PAYMENT          8.1
CONTACT          8.1
FEEDBACK         8.1
SHIPPING         8.0
INVOICE          7.4
DELIVERY         6.7
SUBSCRIPTION     4.1
CANCEL           3.9
Name: proportion, dtype: float64

Test proportions:
label
ACCOUNT         22.1
ORDER           12.9
REFUND          10.7
FEEDBACK         8.1
PAYMENT          8.1
CONTACT          8.1
SHIPPING         8.0
INVOICE          7.4
DELIVERY         6.7
SUBSCRIPTION     4.1
CANCEL           3.8
Name: proportion, dtype: float64


**What to check:** the percentages for each category should be very close
between train and test (e.g., if ACCOUNT is ~20% of train, it should also be
~20% of test). Small differences (under 1%) are normal.


## Step 5: Save the three splits to disk

We save these as CSV files so the next notebook (`04_model_training.ipynb`)
can load them directly, without repeating the cleaning/splitting steps.


In [6]:
import os
os.makedirs("../data/processed", exist_ok=True)

train_df.to_csv("../data/processed/train.csv", index=False)
val_df.to_csv("../data/processed/val.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

print("Saved train.csv, val.csv, test.csv to data/processed/")

Saved train.csv, val.csv, test.csv to data/processed/
